# 초구 예측 — model_20 (3단계: 위치 정규분포 중심·퍼짐을 상황별로, 독립 가정 2)

- 0장 0-1 (2019): 정규분포 중심·퍼짐을 상황 변수로 옮기면 상황 보정 전 24칸 +4.5 · 좌표 NLL +5.8 ×1000, 24칸 상황 보정을 얹은 뒤에는 24칸 +1.2 (경계)
  - 지금 24칸 상황 보정은 좌표를 안 바꿈 -> 좌표 이득은 남을 것
- 모델: model_18에 **정규분포 중심(좌우·높이)·퍼짐의 상황 이동**을 더하고, 그 위에 지금처럼 24칸 상황 보정 (사용자 결정 (나))
  - 이동 = LightGBM 회귀 3개 (표준화 잔차 ex·ez, (ex² + ez²)/2), 변수는 model_14 상황 9개 + 구종·손, 잎 7 · 잎당 최소 20 · λ 1000 · lr 0.05
  - 트리 수는 0장에서 2018 안 조기 종료로 정한 값 (좌우 82 · 높이 33 · 퍼짐 103, 2019로 고르지 않음), 퍼짐 배수는 0.5–2로 자름
  - 월 단위 재학습 (이동 3개 + 24칸 상황 보정 모두, 그 달 첫 경기 전날까지 2018부터)
  - 24칸 상황 보정 학습 공의 출발점은 그 달 이동 모델로 옮긴 확률 (학습 공 자기 예측이라 조금 유리할 수 있음)
- 구종·위치 정규분포 파라미터는 model_18 그대로
- 비교 대상: model_18

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):  # model/ 폴더에서 열면 한 단계 위(데이터·fp_common·results 있는 곳)로
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc

MODEL = "model_20"

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

초구 50,709개 (검증 13,458 / 평가 12,615)


## 0. 모델 전 확인

### 0-1. 정규분포 중심·퍼짐을 상황별로 옮기기

- 출발: model_18 위치 정규분포 (실제 구종, 절대 높이), 2018·2019
- 표준화 잔차 ex = (실제 좌우 − 예측 중심) / 예측 SD, ez 높이 같은 방식
- LightGBM 회귀 (변수 = model_14 상황 9개 + 구종 원-핫·손, 잎 7 · 잎당 최소 20 · λ 1000 · lr 0.05, 학습 2018, 트리 수는 2018 경기 20% 조기 종료)
  - **중심**: ex, ez 각각 예측 -> 중심을 (예측 ex × SD, 예측 ez × SD)만큼 옮김
  - **퍼짐**: (ex² + ez²) / 2 예측 (보정이 맞으면 평균 1) -> 분산·공분산에 곱함 (0.5–2로 자름)
- 비교 (2019, 상황 보정 전): 좌표 NLL, 거리, 24칸 log loss / 그다음 24칸 상황 보정(시즌 단위, model_14 설정)을 얹은 뒤에도 남는지
- 이득 ×1000 = 기본 − 후보 (양수 = 좋아짐), 경기 단위 95% 구간

In [2]:
# ② 위치 (model_18 값) 정규분포: model_13 값 (고정)
LOC_REF, LOC_REF_HL = "decay", 173.0  # 기준점: 분석 대상 투수 구종×손, 전날까지 반감 173구
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 30.0  # 투수 기록 반감기, 투수×구종 당김 k, 투수×구종×손 당김 k
LOC_SEASON_F = 0.707  # 시즌 초 배수
LOC_AC = (None, 12.5, 17.3)  # 모든 카운트 위치 성향 (잔차 기준 반감 None = 누적, δ 반감, δ 당김 k)
LOC_VAR = (200.0, 9.74, False, 316.0)  # 분산 따로 (다른 카운트 퍼짐 반감, k_var, unc, 투수 안 분산 당김 k0)
# ② 위치 (model_18 값) 상황 보정 (model_14): 24칸 LightGBM, 출발점 = model_13 위치 log 확률
LOC_SIT = ["base1", "base2", "base3", "first_open", "sdiff", "inning", "out_cn", "home", "batorder"]  # 상황 변수 (0-3 묶음 4)
LOC_FEAT = CLASSES + ["same_side"] + LOC_SIT  # 구종 원-핫·타자 손 + 상황
LOC_TRAIN_FIRST = 2018  # 위치 보정 학습 첫 해 (2017은 model_13 위치 없음)
LOC_LGB, LOC_ROUNDS = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}, 168  # model_14 상황 보정 LightGBM (고정)
LOC_RETRAIN = "month"  # 상황 보정 재학습 주기 (model_16): 그 달 첫 경기 전날까지 데이터로 다시 학습
LOC_HEIGHT = "abs"  # 위치 높이 표현 (model_18): 절대 높이 cm (지금까지는 존 비율 z_norm)
ZONE_M = 32
ZONE_M = 32
# 24칸 적분 함수: fc.zone_from_gauss_cov(g, q, m=32) (model_03부터)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 분석 대상 투수 구종×손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_sn(q, half_life, season_f, cache=True):  # 투수×구종, 투수×구종×손 전날까지 초구 위치 합계 S·개수 N 두 쌍 (시즌 초 배수 반영)
    return [fc.lookup(fc.prior_table(fp, keys, L_COLS, half_life, season_f=season_f, cache=cache), q, keys, L_COLS, season_f=season_f)
            for keys in (K1, K2)]

def shift_mean(M, dx, dz):  # 모멘트 M (n, 29)의 평균만 (dx, dz) 옮김, 분산·공분산 그대로 (24칸 원-핫 열은 정규분포에 안 쓰여 그대로)
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    nx, nz = mx + dx, mz + dz
    M[:, 2] += nx ** 2 - mx ** 2  # E[x²] = 분산 + 평균²
    M[:, 3] += nz ** 2 - mz ** 2
    M[:, 4] += nx * nz - mx * mz  # E[x·z] = 공분산 + 평균 곱
    M[:, 0], M[:, 1] = nx, nz
    return M

# 0-0 아닌 공 (구종 의심 제외): 모든 카운트 위치 성향 재료
MO = m[~m["label_doubt"] & ~(m["ball_before"].eq(0) & m["strike_before"].eq(0))].copy()
MO["x"], MO["zn"] = MO["plate_x_in_CM"], MO["z_norm"]  # 좌우 cm (타자 기준), 높이 비율 (존 아랫선 0, 윗선 1)
MO["cnt"] = MO["ball_before"].astype(int).astype(str) + "-" + MO["strike_before"].astype(int).astype(str)  # 카운트
CK = ["cnt", "pitch_type", "same_side"]  # 카운트 효과 그룹
_MO_RES = {}

def mo_resid(ref_hl):  # 0-0 아닌 공마다 위치 − 분석 대상 투수 그 카운트×구종×손 전날까지 평균 (반감 ref_hl구) -> 잔차 표 (rx 좌우 cm, rz 높이 비율)
    if ref_hl not in _MO_RES:
        S, N = fc.lookup(fc.prior_table(MO, CK, ["x", "zn"], ref_hl, cache=False), MO, CK, ["x", "zn"])
        ref = np.where(N > 0, S / np.maximum(N, 1e-12), np.nan)  # 그 카운트×구종×손 기록이 없으면 빈 값 (셈에서 빠짐)
        _MO_RES[ref_hl] = MO[K2 + ["game_date", "gameID", "pit_no"]].assign(rx=MO["x"].to_numpy() - ref[:, 0], rz=MO["zn"].to_numpy() - ref[:, 1])
    return _MO_RES[ref_hl]

def ac_sn(q, ref_hl, hl_b, season_f, cache=True):  # 공마다 투수×구종×손 다른 카운트 잔차 전날까지 합 S·개수 N (n, 2)
    d = mo_resid(ref_hl)
    return fc.lookup(fc.prior_table(d, K2, ["rx", "rz"], hl_b, season_f=season_f, cache=cache), q, K2, ["rx", "rz"], season_f=season_f)

MCOLS = ["x", "zn", "xx", "zz", "xz"]  # 위치 모멘트 (좌우, 높이 비율, 제곱, 곱)
_MO_MOM, _WT = {}, {}

def mo_mom(ref_hl):  # 다른 카운트 잔차 모멘트 표 (x = 좌우 잔차, zn = 높이 비율 잔차, 제곱·곱)
    if ref_hl not in _MO_MOM:
        d = mo_resid(ref_hl)
        _MO_MOM[ref_hl] = d.assign(xx=d["rx"] ** 2, zz=d["rz"] ** 2, xz=d["rx"] * d["rz"]).rename(columns={"rx": "x", "rz": "zn"})
    return _MO_MOM[ref_hl]

def within_tab(name, data, year):  # 구종×손별 투수 안 분산·공분산 (지난 시즌까지 투수×구종×손 분산의 (공 수 − 1) 가중 평균) 표와 전체 값
    if (name, year) not in _WT:
        past = data[data["game_date"].dt.year < year].dropna(subset=["x", "zn"])
        g = past.groupby(K2)
        n = g.size()
        w = (n - 1).clip(lower=0)
        c = pd.DataFrame({"vx": g["x"].var(), "vz": g["zn"].var(),
                          "cxz": (g["xz"].mean() - g["x"].mean() * g["zn"].mean()) * n / (n - 1).clip(lower=1)}).fillna(0)  # 표본 분산·공분산
        num = c.mul(w, axis=0).groupby(level=["pitch_type", "same_side"]).sum()
        wg = w.groupby(level=["pitch_type", "same_side"]).sum()  # 구종×손별 자유도 합 (표본 크기)
        _WT[(name, year)] = (num.div(wg, axis=0), c.mul(w, axis=0).sum() / w.sum(), wg)
    return _WT[(name, year)]

def v0_of(q, name, data, year, k0=0.0):  # 공마다 그 구종×손 투수 안 분산·공분산 (n, 3): 자유도 합이 적으면 전체 구종 값 쪽으로 k0만큼 당김
    tab, allv, wg = within_tab(name, data, year)
    idx = pd.MultiIndex.from_frame(q[KR])
    a = allv.to_numpy()[None, :]  # 전체 구종 투수 안 분산
    v = tab.reindex(idx).to_numpy()
    n = wg.reindex(idx).fillna(0).to_numpy()[:, None]  # 그 구종×손 자유도 합
    v = np.where(np.isnan(v), a, v)
    return np.where(n + k0 > 0, (n * v + k0 * a) / np.maximum(n + k0, 1e-12), a)  # 예: 너클 초구 3구(자유도 2)면 거의 전체 값

def own_var(q, year, ref_hl, hl_var, season_f, k0=0.0, cache=True):  # 공마다 다른 카운트 퍼짐 -> 초구 단위로 바꾼 자기 분산·공분산 (n, 3), (가중) 공 수 (n,), V0 (n, 3)
    data = mo_mom(ref_hl)
    S, N = fc.lookup(fc.prior_table(data, K2, MCOLS, hl_var, season_f=season_f, cache=cache), q, K2, MCOLS, season_f=season_f)
    e = S / np.maximum(N, 1e-12)
    v = np.column_stack([e[:, 2] - e[:, 0] ** 2, e[:, 3] - e[:, 1] ** 2, e[:, 4] - e[:, 0] * e[:, 1]])  # 자기 잔차 분산·공분산
    V0 = v0_of(q, "초구", fp, year, k0)
    V0m = v0_of(q, ("다른 카운트", ref_hl), data, year, k0)
    r = np.sqrt(np.maximum(v[:, :2], 0) / V0m[:, :2])  # 좌우·높이 SD 비율 (자기 / 투수 안)
    v = np.column_stack([V0[:, 0] * r[:, 0] ** 2, V0[:, 1] * r[:, 1] ** 2, v[:, 2] / np.sqrt(V0m[:, 0] * V0m[:, 1]) * np.sqrt(V0[:, 0] * V0[:, 1])])
    n = N[:, 0]
    return np.where(n[:, None] > 0, np.nan_to_num(v), 0.0), n, V0

def apply_var(M, v, n, V0, k_var, unc, N2, k_hand):  # 모멘트 M의 분산·공분산을 따로 당긴 값으로 바꿈 (평균 그대로)
    V = (n[:, None] * v + k_var * V0) / (n[:, None] + k_var)  # 투수 안 분산 쪽으로 k_var만큼
    if unc:
        V = V * (1 + 1 / (N2[:, :1] + k_hand))  # 평균을 덜 알수록 넓힘
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    M[:, 2], M[:, 3], M[:, 4] = V[:, 0] + mx ** 2, V[:, 1] + mz ** 2, V[:, 2] + mx * mz  # 분산 -> 모멘트
    return M

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None):  # 위치 모멘트 (n, 29)
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    if ac is not None:  # 모든 카운트 위치 성향 (model_12)
        S, N = ac_sn(q, ac[0], ac[1], season_f)
        d = S / (N + ac[2])
        P = shift_mean(P, d[:, 0], d[:, 1])
    (S1, N1), (S2, N2) = loc_sn(q, half_life, season_f)
    M = fc.shrink(S2, N2, fc.shrink(S1, N1, P, k_type), k_hand)  # 투수×구종 -> 투수×구종×손
    if var is not None:  # 분산 따로 (model_13): var = (hl_var, k_var, unc, k0)
        v, n, V0 = own_var(q, year, None if ac is None else ac[0], var[0], season_f, var[3])
        M = apply_var(M, v, n, V0, var[1], var[2], N2, k_hand)
    return M

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None, cov=True):  # 구종이 t일 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)
    g = gauss_abs(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl, season_f, ac, var), qt)  # 절대 높이 (model_18)
    return zone_probs(g, qt, cov), g


# 높이 표현 (model_18): 절대 높이 cm, 상수는 2017–18 값 (2019·2020 모두 같은 상수, 미래 정보 없음)
past_h = fp["GYEAR"] < fc.VAL_YEAR
H_BAR = float((fp.loc[past_h, "TOP_SZ_CM"] - fp.loc[past_h, "BOTTOM_SZ_CM"]).mean())  # 평균 존 높이
BOT_BAR = float(fp.loc[past_h, "BOTTOM_SZ_CM"].mean())  # 평균 아랫선
fp["zn"] = (fp["CROSS_PLATE_Z_CM"] - BOT_BAR) / H_BAR  # 높이 열을 절대 높이 표현으로 (구종은 이 열을 안 씀)
fp["zz"], fp["xz"] = fp["zn"] ** 2, fp["x"] * fp["zn"]
MO["zn"] = (MO["CROSS_PLATE_Z_CM"] - BOT_BAR) / H_BAR  # 다른 카운트 공도 같은 표현
for cache in (fc._TABLES, _MO_RES, _MO_MOM, _WT):
    cache.clear()  # 높이 열이 바뀌어 누적표를 새로 (prior_table 캐시는 id(data) 키라 같은 객체면 옛 표를 씀)

def gauss_abs(M, q):  # 모멘트 (x, zn, xx, zz, xz) -> 2차원 정규분포, 높이는 절대 높이로 되돌림 (타자 존 무관)
    mx, mz, xx, zz, xz = M[:, :5].T
    vx = np.maximum(xx - mx ** 2, 4.0)
    vz = np.maximum(zz - mz ** 2, 1e-3) * H_BAR ** 2
    c = np.clip((xz - mx * mz) * H_BAR, -0.99 * np.sqrt(vx * vz), 0.99 * np.sqrt(vx * vz))
    return mx, BOT_BAR + mz * H_BAR, vx, vz, c


LOC17 = dict(half_life=LOC_HL, k_type=K_TYPE, k_hand=K_HAND, ref=LOC_REF, ref_hl=LOC_REF_HL, season_f=LOC_SEASON_F, ac=LOC_AC, var=LOC_VAR)
from scipy.special import log_softmax
yr_l = fp["GYEAR"].to_numpy()
G = {}
for yr in (2018, fc.VAL_YEAR):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P, g = loc_given_type(fp[k], yr, fp.loc[k, "pitch_type"].to_numpy(), **LOC17)
    G[yr] = (np.flatnonzero(k), P, g)
i19, P19, g19 = G[fc.VAL_YEAR]
ll19 = -np.log(np.clip(P19[np.arange(len(i19)), y_zone[i19]], 1e-12, None))
print(f"model_18 위치 (상황 보정 전) 2019 24칸 {ll19.mean():.5f} (model_18 0장 C 2.87214와 같아야 함)")
assert abs(ll19.mean() - 2.87214) < 2e-5

SIT_L = fc.situation_features(m, fp)
XLOC = pd.concat([fp[CLASSES + ["same_side"]], SIT_L, fc.opponent_features(m, fp)[["home"]]], axis=1)
XLOC["first_open"] = (((SIT_L["base2"] + SIT_L["base3"]) > 0) & SIT_L["base1"].eq(0)).astype(int)
XLOC = XLOC[LOC_FEAT]
REG = dict(objective="regression", learning_rate=0.05, num_leaves=7, min_data_in_leaf=20, lambda_l2=1000.0, feature_fraction=0.9,
           bagging_fraction=0.8, bagging_freq=1, verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)
itr = G[2018][0]
gid_all = fp["gameID"].to_numpy()
g18 = np.unique(gid_all[itr])
es = np.isin(gid_all[itr], np.random.default_rng(1).choice(g18, len(g18) // 5, replace=False))

def resid(idx, g):  # 표준화 잔차 ex, ez, (ex² + ez²)/2
    mx, mz, vx, vz, c = g
    ex = (fp["x"].to_numpy()[idx] - mx) / np.sqrt(vx)
    ez = (fp["CROSS_PLATE_Z_CM"].to_numpy()[idx] - mz) / np.sqrt(vz)
    return ex, ez, (ex ** 2 + ez ** 2) / 2

E18, E19 = resid(itr, G[2018][2]), resid(i19, g19)
def fit_reg(y):  # 2018 학습 (경기 20%로 조기 종료) -> 2019 예측
    t_, v_ = np.flatnonzero(~es), np.flatnonzero(es)
    dtr = lgb.Dataset(XLOC.iloc[itr[t_]], y[t_])
    dva = lgb.Dataset(XLOC.iloc[itr[v_]], y[v_], reference=dtr)
    bst = lgb.train(REG, dtr, 3000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
    return bst.predict(XLOC.iloc[i19], num_iteration=bst.best_iteration), bst.best_iteration

pex, nx = fit_reg(E18[0])
pez, nz = fit_reg(E18[1])
psc, ns = fit_reg(E18[2])
psc = np.clip(psc, 0.5, 2.0)
print(f"트리 수: 중심 좌우 {nx} · 높이 {nz} · 퍼짐 {ns} / 2019 예측 범위: ex {pex.min():.3f} ~ {pex.max():.3f}, ez {pez.min():.3f} ~ {pez.max():.3f}, 퍼짐 배수 {psc.min():.2f} ~ {psc.max():.2f}")

def moved(g, shift=True, spread=True):
    mx, mz, vx, vz, c = g
    if shift:
        mx, mz = mx + pex * np.sqrt(vx), mz + pez * np.sqrt(vz)
    if spread:
        vx, vz, c = vx * psc, vz * psc, c * psc
    return mx, mz, vx, vz, c

q19 = fp.iloc[i19]
yz19 = y_zone[i19]
def scores(g):
    P = fc.zone_from_gauss_cov(g, q19, ZONE_M)
    mx, mz = g[0], g[1]
    return P, -np.log(np.clip(P[np.arange(len(i19)), yz19], 1e-12, None)), -fc.log_dens(g, q19), np.hypot(q19["x"].to_numpy() - mx, q19["CROSS_PLATE_Z_CM"].to_numpy() - mz)

def game_ci(d, B=1000):
    s = pd.DataFrame({"g": gid_all[i19], "d": d}).groupby("g")["d"].agg(["sum", "count"])
    ii = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[ii].sum(1) / s["count"].to_numpy()[ii].sum(1), [2.5, 97.5])

P0, l0, c0, d0 = scores(g19)
VAR = {"중심만": moved(g19, True, False), "퍼짐만": moved(g19, False, True), "중심 + 퍼짐": moved(g19, True, True)}
rows, PV = [], {"기본 (model_18 위치)": P0}
rows.append({"후보": "기본 (model_18 위치)", "24칸 ll": l0.mean(), "좌표 NLL": c0.mean(), "거리 cm": d0.mean()})
for nm, g in VAR.items():
    P, l, c, dd = scores(g)
    PV[nm] = P
    r = {"후보": nm, "24칸 ll": l.mean(), "좌표 NLL": c.mean(), "거리 cm": dd.mean()}
    for lab, a, b in [("24칸", l0, l), ("좌표 NLL", c0, c)]:
        lo, hi = game_ci(a - b)
        r[f"{lab} 이득 ×1000"] = (a - b).mean() * 1000
        r[f"{lab} 95%"] = f"{lo * 1000:+.2f} ~ {hi * 1000:+.2f}"
    rows.append(r)
print("2019, 상황 보정 전")
display(pd.DataFrame(rows).set_index("후보").round(4))

model_18 위치 (상황 보정 전) 2019 24칸 2.87214 (model_18 0장 C 2.87214와 같아야 함)
트리 수: 중심 좌우 82 · 높이 33 · 퍼짐 103 / 2019 예측 범위: ex -0.134 ~ 0.188, ez -0.072 ~ 0.128, 퍼짐 배수 0.86 ~ 1.27
2019, 상황 보정 전


,24칸 ll,좌표 NLL,거리 cm,24칸 이득 ×1000,24칸 95%,좌표 NLL 이득 ×1000,좌표 NLL 95%
후보,,,,,,,
기본 (model_18 위치),2.8721,9.2229,31.5252,NaN,NaN,NaN,NaN
중심만,2.8683,9.2183,31.4408,3.8344,+2.34 ~ +5.37,4.5895,+2.94 ~ +6.36
퍼짐만,2.8716,9.2217,31.5252,0.5900,-0.27 ~ +1.42,1.1971,+0.08 ~ +2.39
중심 + 퍼짐,2.8677,9.2171,31.4408,4.4584,+2.76 ~ +6.22,5.8219,+3.78 ~ +7.88


#### 0-1-2. 24칸 상황 보정을 얹은 뒤에도 남는지

- 24칸 상황 보정 LightGBM (model_14 설정: 잎 7 · 잎당 최소 20 · λ 1000 · 트리 168, 시즌 단위 학습 2018)을 기본 / 중심 + 퍼짐 위에 각각 얹어 2019 24칸 비교
- 중심 + 퍼짐 쪽은 2018 학습 공에도 같은 옮기기를 적용 (2018 회귀 모델의 학습 공 예측이라 조금 유리할 수 있음)

In [3]:
def zone_corr(P_tr, P_te):  # 2018 출발 확률로 24칸 보정 학습 -> 2019 보정 확률
    init_tr = np.log(np.clip(P_tr, 1e-6, None)); init_tr -= init_tr.mean(1, keepdims=True)
    init_te = np.log(np.clip(P_te, 1e-6, None)); init_te -= init_te.mean(1, keepdims=True)
    p = dict(objective="multiclass", num_class=len(ZONES), learning_rate=0.05, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
             verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4, num_leaves=7, min_data_in_leaf=20, lambda_l2=1000.0)
    bst = lgb.train(p, lgb.Dataset(XLOC.iloc[itr], y_zone[itr], init_score=init_tr), 168)
    return np.exp(log_softmax(bst.predict(XLOC.iloc[i19], raw_score=True) + init_te, axis=1))

# 2018 공의 중심 + 퍼짐 옮기기: 2018 회귀 모델로 2018 공 예측 (학습 공이라 조금 유리)
t_, v_ = np.flatnonzero(~es), np.flatnonzero(es)
def fit_pred18(y):
    dtr = lgb.Dataset(XLOC.iloc[itr[t_]], y[t_]); dva = lgb.Dataset(XLOC.iloc[itr[v_]], y[v_], reference=dtr)
    bst = lgb.train(REG, dtr, 3000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
    return bst.predict(XLOC.iloc[itr], num_iteration=bst.best_iteration)
ex18, ez18, sc18 = fit_pred18(E18[0]), fit_pred18(E18[1]), np.clip(fit_pred18(E18[2]), 0.5, 2.0)
mx, mz, vx, vz, c = G[2018][2]
g18m = (mx + ex18 * np.sqrt(vx), mz + ez18 * np.sqrt(vz), vx * sc18, vz * sc18, c * sc18)
P18m = fc.zone_from_gauss_cov(g18m, fp.iloc[itr], ZONE_M)
Pc0 = zone_corr(G[2018][1], P0)
Pc1 = zone_corr(P18m, PV["중심 + 퍼짐"])
lc0 = -np.log(np.clip(Pc0[np.arange(len(i19)), yz19], 1e-12, None))
lc1 = -np.log(np.clip(Pc1[np.arange(len(i19)), yz19], 1e-12, None))
lo, hi = game_ci(lc0 - lc1)
hsc = lambda P: -np.log(np.clip((P @ fc.GROUP2)[np.arange(len(i19)), pd.Series(fc.ZONE_GROUP[yz19]).map({"H": 0, "S": 1, "C": 2}).to_numpy()], 1e-12, None))
lo2, hi2 = game_ci(hsc(Pc0) - hsc(Pc1))
print(f"2019 24칸 (24칸 상황 보정 후): 기본 {lc0.mean():.5f} / 중심 + 퍼짐 {lc1.mean():.5f}, 이득 ×1000 {(lc0 - lc1).mean() * 1000:.2f} (95% {lo * 1000:.2f} ~ {hi * 1000:.2f})")
print(f"   H/S/C log loss 이득 ×1000 {(hsc(Pc0) - hsc(Pc1)).mean() * 1000:.2f} (95% {lo2 * 1000:.2f} ~ {hi2 * 1000:.2f})")

2019 24칸 (24칸 상황 보정 후): 기본 2.86909 / 중심 + 퍼짐 2.86788, 이득 ×1000 1.20 (95% -0.01 ~ 2.45)
   H/S/C log loss 이득 ×1000 0.06 (95% -0.46 ~ 0.57)


#### 🔍 0-1 결과: 좌표는 좋아지고, 24칸은 대부분 이미 24칸 상황 보정이 가져감
| 2019, 상황 보정 전 | 24칸 이득 ×1000 (95%) | 좌표 NLL 이득 ×1000 (95%) | 거리 |
| --- | --- | --- | --- |
| 중심만 | +3.83 (+2.34 ~ +5.37) | +4.59 (+2.94 ~ +6.36) | 31.53 -> 31.44cm |
| 퍼짐만 | +0.59 (−0.27 ~ +1.42) | +1.20 (+0.08 ~ +2.39) | 그대로 |
| **중심 + 퍼짐** | **+4.46** (+2.76 ~ +6.22) | **+5.82** (+3.78 ~ +7.88) | 31.44cm |

- 회귀 예측 범위: 중심 ex −0.13 ~ +0.19, ez −0.07 ~ +0.13 (SD 단위, 몇 cm 정도), 퍼짐 배수 0.86 ~ 1.27 -> 강한 규제로 작은 보정
- **24칸 상황 보정을 얹은 뒤** (2019, 시즌 단위): 24칸 +1.20 (−0.01 ~ +2.45), H/S/C +0.06 (−0.46 ~ +0.57) -> 24칸 이득의 대부분(4.5 중 3.3)은 24칸 상황 보정과 겹침
- 지금 24칸 상황 보정은 좌표를 안 바꾸므로 **좌표 NLL·거리 이득(+5.8)은 그대로 남을 것** (model_14–18 부트스트랩에서 좌표 차이 0)


## 모델

In [4]:
# 0장이 모델 변수 이름을 덮어쓰지 않았는지 대비: 설정 상수를 다시 정함
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]


## 1. 파라미터

- 모두 model_18 값 고정, 더하는 것: 위치 중심·퍼짐 상황 이동 `LOC_SHIFT`

In [5]:
# ① 구종 출발점 B4: model_06 값 (고정) - 반감기·alpha·hand_k, 시즌 초 배수 (기준점 = 지난 시즌 전부)
B4_P = dict(half_life=35.4, alpha=7.5, hand_k=5.48, season_f=0.25)
# ① LightGBM 변수: A 이번 타석 상황 + 앞 타석 초구 결과 원-핫 (fc.FIRST_CATS)
FEAT = fc.A_COLS + ["prev_first_" + c for c in fc.FIRST_CATS]
# ② 위치 정규분포: model_13 값 (고정)
LOC_REF, LOC_REF_HL = "decay", 173.0  # 기준점: 분석 대상 투수 구종×손, 전날까지 반감 173구
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 30.0  # 투수 기록 반감기, 투수×구종 당김 k, 투수×구종×손 당김 k
LOC_SEASON_F = 0.707  # 시즌 초 배수
LOC_AC = (None, 12.5, 17.3)  # 모든 카운트 위치 성향 (잔차 기준 반감 None = 누적, δ 반감, δ 당김 k)
LOC_VAR = (200.0, 9.74, False, 316.0)  # 분산 따로 (다른 카운트 퍼짐 반감, k_var, unc, 투수 안 분산 당김 k0)
# ② 위치 상황 보정 (model_14): 24칸 LightGBM, 출발점 = model_13 위치 log 확률
LOC_SIT = ["base1", "base2", "base3", "first_open", "sdiff", "inning", "out_cn", "home", "batorder"]  # 상황 변수 (0-3 묶음 4)
LOC_FEAT = CLASSES + ["same_side"] + LOC_SIT  # 구종 원-핫·타자 손 + 상황
LOC_TRAIN_FIRST = 2018  # 위치 보정 학습 첫 해 (2017은 model_13 위치 없음)
LOC_LGB, LOC_ROUNDS = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}, 168  # model_14 상황 보정 LightGBM (고정)
LOC_RETRAIN = "month"  # 상황 보정 재학습 주기 (model_16): 그 달 첫 경기 전날까지 데이터로 다시 학습
LOC_HEIGHT = "abs"  # 위치 높이 표현 (model_18): 절대 높이 cm (지금까지는 존 비율 z_norm)
LOC_SHIFT = {"ex": 82, "ez": 33, "u": 103}  # 위치 중심(좌우·높이)·퍼짐 상황 이동 LightGBM 회귀 트리 수 (model_20, 0장 2018 안 조기 종료)
LOC_SHIFT_LGB = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}  # 이동 회귀 파라미터 (model_14 값)
ZONE_M = 32
# ① LightGBM: model_07 값 (고정)
LGB_P = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}
LGB_ROUNDS = 730
RETRAIN = "month"  # 재학습 주기: 그 달 첫 경기 전날까지 데이터로 다시 학습 (3월은 4월에, 10월은 9월에 합침)
# 구종 보정 학습 해
TRAIN_FIRST = 2017

## 2. ① 구종: B4 + LightGBM 월 단위 재학습 (model_09과 같음)

### 2-1. B4 (model_06·07·09과 같음)

In [6]:
def b4_online(q, year, half_life, alpha, hand_k, season_f):  # model_06 B4 (n, 9): 지난 시즌 리그 비율 -> 투수 (alpha) -> 투수×손 (hand_k), 시즌 초 배수
    first = year == fp["GYEAR"].min()  # 첫 시즌(2017): 지난 기록 없음 -> 그 시즌 비율 (학습 데이터 출발점용)
    ref = fp.loc[fp["GYEAR"] == year if first else fp["GYEAR"] < year, CLASSES].mean().to_numpy()
    out = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        S, N = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life, season_f=season_f), q, keys, CLASSES, season_f=season_f)
        out += [S, N[:, :1]]
    S, N, S2, N2 = out
    return fc.shrink(S2, N2, fc.shrink(S, N, ref, alpha), hand_k)

K = len(CLASSES)
P_B4 = np.full((len(fp), K), np.nan)  # 2017–2020
for yr in range(TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P_B4[k] = b4_online(fp[k], yr, **B4_P)
b4_19 = fc.prob_metrics(P_B4[val], y_type[val])["logloss"]
assert abs(b4_19 - 1.09367) < 1e-5, b4_19  # model_06·07 B4의 2019 log loss
print(f"2019 B4 log loss {b4_19:.5f} = model_06·07 B4 (재현 확인)")
m07 = pd.read_parquet("results/model_07/per_pitch.parquet")  # 첫 달 재현 확인용
m09 = pd.read_parquet("results/model_09/per_pitch.parquet")  # 구종 재현 확인용 (구종은 model_09 그대로)
m13 = pd.read_parquet("results/model_13/per_pitch.parquet")  # 위치 정규분포 재현 확인용
m14 = pd.read_parquet("results/model_14/per_pitch.parquet")  # 참고
m16 = pd.read_parquet("results/model_16/per_pitch.parquet")  # 참고
m18 = pd.read_parquet("results/model_18/per_pitch.parquet")  # 비교 대상

2019 B4 log loss 1.09367 = model_06·07 B4 (재현 확인)


### 2-2. 변수 (model_07과 같음)

- A 이번 타석 상황 + 앞 타석 초구 결과, 미래 정보 검사 다시 통과 확인

In [7]:
X_all = pd.concat([fc.situation_features(m, fp), fc.game_flow_features(m, fp)], axis=1)[FEAT]
assert X_all.notna().all().all()
for fn in (fc.situation_features, fc.game_flow_features):
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(TRAIN_FIRST)], n=150)
    assert bad.empty, bad
display(X_all.describe().T[["mean", "min", "max"]].round(3))

미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개


,mean,min,max
pc,33.394,0.0,124.0
tto,1.592,1.0,5.0
starter,0.690,0.0,1.0
inning,4.548,1.0,12.0
out_cn,0.970,0.0,2.0
base1,0.359,0.0,1.0
base2,0.194,0.0,1.0
base3,0.100,0.0,1.0
batorder,4.811,1.0,9.0
sdiff,0.369,-21.0,19.0


### 2-2. 월 단위 재학습과 구종 확률 (model_09과 같음)

- 확인: 구종 확률 = model_09 결과 파일 (재현)

In [8]:
y_all = fp["pitch_type"].map(fc.TYPE_IDX).to_numpy()
INIT = np.log(np.clip(P_B4, 1e-6, None))
INIT -= INIT.mean(1, keepdims=True)  # 출발점 (공마다 상수는 softmax에 영향 없음)
BASE = dict(objective="multiclass", num_class=K, learning_rate=0.03, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
            verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)  # model_05–07과 같음
yr_all = fp["GYEAR"].to_numpy()
DATE = pd.to_datetime(fp["gameID"].str[:8]).to_numpy()
MON = np.clip(fp["gameID"].str[4:6].astype(int).to_numpy(), 4, 9)  # 재학습 묶음 달: 3월 -> 4월, 10월 -> 9월

def fit_rows(tr):  # 학습 행 tr로 LightGBM (트리 수 고정)
    return lgb.train({**BASE, **LGB_P}, lgb.Dataset(X_all.iloc[tr], y_all[tr], init_score=INIT[tr]), LGB_ROUNDS)

def type_prob(bst, idx):  # B4 출발점 + 트리 보정 -> 구종 확률 (n, 9)
    return np.exp(log_softmax(bst.predict(X_all.iloc[idx], raw_score=True) + INIT[idx], axis=1))

P_TYPE_ALL = np.full((len(fp), K), np.nan)
BST, plan = {}, []
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    iy = np.flatnonzero(yr_all == yr)
    for mon in np.unique(MON[iy]):
        te = iy[MON[iy] == mon]
        start = DATE[te].min()  # 그 달 첫 경기일
        tr = np.flatnonzero((yr_all >= TRAIN_FIRST) & (DATE < start))
        assert DATE[tr].max() < start  # 미래 정보 없음
        BST[(yr, mon)] = bst = fit_rows(tr)
        P_TYPE_ALL[te] = type_prob(bst, te)
        plan.append({"연도": yr, "월": mon, "예측 초구": len(te), "학습 초구": len(tr), "그 시즌 학습 초구": int((yr_all[tr] == yr).sum()),
                     "학습 끝": str(pd.Timestamp(DATE[tr].max()).date()), "예측 시작": str(pd.Timestamp(start).date())})
display(pd.DataFrame(plan).set_index(["연도", "월"]))
P11 = {yr: fc.align(m07, fp, yr)[fc.P_TYPE].to_numpy() for yr in (fc.VAL_YEAR, fc.TEST_YEAR)}  # model_07 구종 확률 (fp 그 해 순서)
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:  # 첫 달 = 시즌 단위 (model_07 재현)
    k = MON[msk] == MON[msk].min()
    assert np.allclose(P_TYPE_ALL[msk][k], P11[yr][k], atol=1e-6)
print("첫 달 구종 확률 = model_07 (재현 확인)")
rows = {}
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    for nm, P in [("B4", P_B4[msk]), ("model_07", P11[yr]), (MODEL, P_TYPE_ALL[msk])]:
        r = fc.prob_metrics(P, y_type[msk])
        rows[(yr, nm)] = {"log loss": r["logloss"], "top1": r["top1"], "top2": r["top2"]}
display(pd.DataFrame(rows).T.round(4))
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    assert np.allclose(P_TYPE_ALL[msk], fc.align(m09, fp, yr)[fc.P_TYPE].to_numpy(), atol=1e-6), f"{yr} 구종 확률이 model_09과 다름"
print("구종 확률 = model_09 (재현 확인)")

예측 초구  학습 초구  그 시즌 학습 초구        학습 끝       예측 시작
연도   월                                                  
2019 4   2568  24636           0  2018-10-14  2019-03-23
     5   2403  27204        2568  2019-04-30  2019-05-01
     6   2493  29607        4971  2019-05-31  2019-06-01
     7   1791  32100        7464  2019-06-30  2019-07-02
     8   2362  33891        9255  2019-07-31  2019-08-01
     9   1841  36253       11617  2019-08-31  2019-09-01
2020 5   2264  38094           0  2019-10-01  2020-05-05
     6   2379  40358        2264  2020-05-31  2020-06-02
     7   2233  42737        4643  2020-06-30  2020-07-01
     8   1985  44970        6876  2020-07-31  2020-08-01
     9   3754  46955        8861  2020-08-30  2020-09-01

첫 달 구종 확률 = model_07 (재현 확인)


log loss    top1    top2
2019 B4          1.0937  0.5279  0.8159
     model_07    1.0621  0.5380  0.8174
     model_20    1.0613  0.5385  0.8177
2020 B4          1.0916  0.5264  0.8179
     model_07    1.0570  0.5340  0.8209
     model_20    1.0561  0.5332  0.8218

구종 확률 = model_09 (재현 확인)


## 3. ② 위치: 절대 높이 정규분포 + 중심·퍼짐 상황 이동 + 24칸 상황 보정 (월 단위)

### 3-1. 위치 정규분포 (model_18와 같음)

- 실제 구종 기준 정규분포와 24칸 확률 (2018–2020) -> 2019 24칸이 model_18 상황 보정 전 값(2.87214)과 같은지 확인

In [10]:
# 24칸 적분 함수: fc.zone_from_gauss_cov(g, q, m=32) (model_03부터)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 분석 대상 투수 구종×손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_sn(q, half_life, season_f, cache=True):  # 투수×구종, 투수×구종×손 전날까지 초구 위치 합계 S·개수 N 두 쌍 (시즌 초 배수 반영)
    return [fc.lookup(fc.prior_table(fp, keys, L_COLS, half_life, season_f=season_f, cache=cache), q, keys, L_COLS, season_f=season_f)
            for keys in (K1, K2)]

def shift_mean(M, dx, dz):  # 모멘트 M (n, 29)의 평균만 (dx, dz) 옮김, 분산·공분산 그대로 (24칸 원-핫 열은 정규분포에 안 쓰여 그대로)
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    nx, nz = mx + dx, mz + dz
    M[:, 2] += nx ** 2 - mx ** 2  # E[x²] = 분산 + 평균²
    M[:, 3] += nz ** 2 - mz ** 2
    M[:, 4] += nx * nz - mx * mz  # E[x·z] = 공분산 + 평균 곱
    M[:, 0], M[:, 1] = nx, nz
    return M

# 0-0 아닌 공 (구종 의심 제외): 모든 카운트 위치 성향 재료
MO = m[~m["label_doubt"] & ~(m["ball_before"].eq(0) & m["strike_before"].eq(0))].copy()
MO["x"], MO["zn"] = MO["plate_x_in_CM"], MO["z_norm"]  # 좌우 cm (타자 기준), 높이 비율 (존 아랫선 0, 윗선 1)
MO["cnt"] = MO["ball_before"].astype(int).astype(str) + "-" + MO["strike_before"].astype(int).astype(str)  # 카운트
CK = ["cnt", "pitch_type", "same_side"]  # 카운트 효과 그룹
_MO_RES = {}

def mo_resid(ref_hl):  # 0-0 아닌 공마다 위치 − 분석 대상 투수 그 카운트×구종×손 전날까지 평균 (반감 ref_hl구) -> 잔차 표 (rx 좌우 cm, rz 높이 비율)
    if ref_hl not in _MO_RES:
        S, N = fc.lookup(fc.prior_table(MO, CK, ["x", "zn"], ref_hl, cache=False), MO, CK, ["x", "zn"])
        ref = np.where(N > 0, S / np.maximum(N, 1e-12), np.nan)  # 그 카운트×구종×손 기록이 없으면 빈 값 (셈에서 빠짐)
        _MO_RES[ref_hl] = MO[K2 + ["game_date", "gameID", "pit_no"]].assign(rx=MO["x"].to_numpy() - ref[:, 0], rz=MO["zn"].to_numpy() - ref[:, 1])
    return _MO_RES[ref_hl]

def ac_sn(q, ref_hl, hl_b, season_f, cache=True):  # 공마다 투수×구종×손 다른 카운트 잔차 전날까지 합 S·개수 N (n, 2)
    d = mo_resid(ref_hl)
    return fc.lookup(fc.prior_table(d, K2, ["rx", "rz"], hl_b, season_f=season_f, cache=cache), q, K2, ["rx", "rz"], season_f=season_f)

MCOLS = ["x", "zn", "xx", "zz", "xz"]  # 위치 모멘트 (좌우, 높이 비율, 제곱, 곱)
_MO_MOM, _WT = {}, {}

def mo_mom(ref_hl):  # 다른 카운트 잔차 모멘트 표 (x = 좌우 잔차, zn = 높이 비율 잔차, 제곱·곱)
    if ref_hl not in _MO_MOM:
        d = mo_resid(ref_hl)
        _MO_MOM[ref_hl] = d.assign(xx=d["rx"] ** 2, zz=d["rz"] ** 2, xz=d["rx"] * d["rz"]).rename(columns={"rx": "x", "rz": "zn"})
    return _MO_MOM[ref_hl]

def within_tab(name, data, year):  # 구종×손별 투수 안 분산·공분산 (지난 시즌까지 투수×구종×손 분산의 (공 수 − 1) 가중 평균) 표와 전체 값
    if (name, year) not in _WT:
        past = data[data["game_date"].dt.year < year].dropna(subset=["x", "zn"])
        g = past.groupby(K2)
        n = g.size()
        w = (n - 1).clip(lower=0)
        c = pd.DataFrame({"vx": g["x"].var(), "vz": g["zn"].var(),
                          "cxz": (g["xz"].mean() - g["x"].mean() * g["zn"].mean()) * n / (n - 1).clip(lower=1)}).fillna(0)  # 표본 분산·공분산
        num = c.mul(w, axis=0).groupby(level=["pitch_type", "same_side"]).sum()
        wg = w.groupby(level=["pitch_type", "same_side"]).sum()  # 구종×손별 자유도 합 (표본 크기)
        _WT[(name, year)] = (num.div(wg, axis=0), c.mul(w, axis=0).sum() / w.sum(), wg)
    return _WT[(name, year)]

def v0_of(q, name, data, year, k0=0.0):  # 공마다 그 구종×손 투수 안 분산·공분산 (n, 3): 자유도 합이 적으면 전체 구종 값 쪽으로 k0만큼 당김
    tab, allv, wg = within_tab(name, data, year)
    idx = pd.MultiIndex.from_frame(q[KR])
    a = allv.to_numpy()[None, :]  # 전체 구종 투수 안 분산
    v = tab.reindex(idx).to_numpy()
    n = wg.reindex(idx).fillna(0).to_numpy()[:, None]  # 그 구종×손 자유도 합
    v = np.where(np.isnan(v), a, v)
    return np.where(n + k0 > 0, (n * v + k0 * a) / np.maximum(n + k0, 1e-12), a)  # 예: 너클 초구 3구(자유도 2)면 거의 전체 값

def own_var(q, year, ref_hl, hl_var, season_f, k0=0.0, cache=True):  # 공마다 다른 카운트 퍼짐 -> 초구 단위로 바꾼 자기 분산·공분산 (n, 3), (가중) 공 수 (n,), V0 (n, 3)
    data = mo_mom(ref_hl)
    S, N = fc.lookup(fc.prior_table(data, K2, MCOLS, hl_var, season_f=season_f, cache=cache), q, K2, MCOLS, season_f=season_f)
    e = S / np.maximum(N, 1e-12)
    v = np.column_stack([e[:, 2] - e[:, 0] ** 2, e[:, 3] - e[:, 1] ** 2, e[:, 4] - e[:, 0] * e[:, 1]])  # 자기 잔차 분산·공분산
    V0 = v0_of(q, "초구", fp, year, k0)
    V0m = v0_of(q, ("다른 카운트", ref_hl), data, year, k0)
    r = np.sqrt(np.maximum(v[:, :2], 0) / V0m[:, :2])  # 좌우·높이 SD 비율 (자기 / 투수 안)
    v = np.column_stack([V0[:, 0] * r[:, 0] ** 2, V0[:, 1] * r[:, 1] ** 2, v[:, 2] / np.sqrt(V0m[:, 0] * V0m[:, 1]) * np.sqrt(V0[:, 0] * V0[:, 1])])
    n = N[:, 0]
    return np.where(n[:, None] > 0, np.nan_to_num(v), 0.0), n, V0

def apply_var(M, v, n, V0, k_var, unc, N2, k_hand):  # 모멘트 M의 분산·공분산을 따로 당긴 값으로 바꿈 (평균 그대로)
    V = (n[:, None] * v + k_var * V0) / (n[:, None] + k_var)  # 투수 안 분산 쪽으로 k_var만큼
    if unc:
        V = V * (1 + 1 / (N2[:, :1] + k_hand))  # 평균을 덜 알수록 넓힘
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    M[:, 2], M[:, 3], M[:, 4] = V[:, 0] + mx ** 2, V[:, 1] + mz ** 2, V[:, 2] + mx * mz  # 분산 -> 모멘트
    return M

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None):  # 위치 모멘트 (n, 29)
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    if ac is not None:  # 모든 카운트 위치 성향 (model_12)
        S, N = ac_sn(q, ac[0], ac[1], season_f)
        d = S / (N + ac[2])
        P = shift_mean(P, d[:, 0], d[:, 1])
    (S1, N1), (S2, N2) = loc_sn(q, half_life, season_f)
    M = fc.shrink(S2, N2, fc.shrink(S1, N1, P, k_type), k_hand)  # 투수×구종 -> 투수×구종×손
    if var is not None:  # 분산 따로 (model_13): var = (hl_var, k_var, unc, k0)
        v, n, V0 = own_var(q, year, None if ac is None else ac[0], var[0], season_f, var[3])
        M = apply_var(M, v, n, V0, var[1], var[2], N2, k_hand)
    return M

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None, cov=True):  # 구종이 t일 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)
    g = gauss_abs(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl, season_f, ac, var), qt)  # 절대 높이 (model_18)
    return zone_probs(g, qt, cov), g


# 높이 표현 (model_18): 절대 높이 cm, 상수는 2017–18 값 (2019·2020 모두 같은 상수, 미래 정보 없음)
past_h = fp["GYEAR"] < fc.VAL_YEAR
H_BAR = float((fp.loc[past_h, "TOP_SZ_CM"] - fp.loc[past_h, "BOTTOM_SZ_CM"]).mean())  # 평균 존 높이
BOT_BAR = float(fp.loc[past_h, "BOTTOM_SZ_CM"].mean())  # 평균 아랫선
fp["zn"] = (fp["CROSS_PLATE_Z_CM"] - BOT_BAR) / H_BAR  # 높이 열을 절대 높이 표현으로 (구종은 이 열을 안 씀)
fp["zz"], fp["xz"] = fp["zn"] ** 2, fp["x"] * fp["zn"]
MO["zn"] = (MO["CROSS_PLATE_Z_CM"] - BOT_BAR) / H_BAR  # 다른 카운트 공도 같은 표현
for cache in (fc._TABLES, _MO_RES, _MO_MOM, _WT):
    cache.clear()  # 높이 열이 바뀌어 누적표를 새로 (prior_table 캐시는 id(data) 키라 같은 객체면 옛 표를 씀)

def gauss_abs(M, q):  # 모멘트 (x, zn, xx, zz, xz) -> 2차원 정규분포, 높이는 절대 높이로 되돌림 (타자 존 무관)
    mx, mz, xx, zz, xz = M[:, :5].T
    vx = np.maximum(xx - mx ** 2, 4.0)
    vz = np.maximum(zz - mz ** 2, 1e-3) * H_BAR ** 2
    c = np.clip((xz - mx * mz) * H_BAR, -0.99 * np.sqrt(vx * vz), 0.99 * np.sqrt(vx * vz))
    return mx, BOT_BAR + mz * H_BAR, vx, vz, c


LOC17 = dict(half_life=LOC_HL, k_type=K_TYPE, k_hand=K_HAND, ref=LOC_REF, ref_hl=LOC_REF_HL, season_f=LOC_SEASON_F, ac=LOC_AC, var=LOC_VAR)
GT = np.full((len(fp), 5), np.nan)  # 실제 구종 기준 정규분포 (중심 좌우, 중심 높이, 분산 좌우, 분산 높이, 공분산)
PZ_TRUE = np.full((len(fp), len(ZONES)), np.nan)
for yr in range(LOC_TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P, g = loc_given_type(fp[k], yr, fp.loc[k, "pitch_type"].to_numpy(), **LOC17)
    PZ_TRUE[k], GT[k] = P, np.column_stack(g)
ll_ = -np.log(np.clip(PZ_TRUE[val][np.arange(val.sum()), y_zone[val]], 1e-12, None))
assert abs(ll_.mean() - 2.87214) < 2e-5, ll_.mean()
print(f"위치 정규분포 (상황 보정 전) 2019 24칸 {ll_.mean():.5f} = model_18 (재현 확인)")

위치 정규분포 (상황 보정 전) 2019 24칸 2.87214 = model_18 (재현 확인)


### 3-2. 중심·퍼짐 상황 이동 + 24칸 상황 보정 (월 단위)

- 달마다 학습 공(2018부터 그 달 첫 경기 전날까지)으로: ① 이동 회귀 3개 학습 (트리 수 고정) -> ② 학습 공의 정규분포를 옮겨 24칸 확률 -> ③ 그 확률을 출발점으로 24칸 상황 보정 학습 (트리 168)
- 예측: 그 달 이동 모델로 정규분포를 옮김 -> 24칸 적분 -> 그 달 24칸 상황 보정, 좌표 점수는 옮긴 정규분포로

In [11]:
from scipy.special import log_softmax
for fn in (fc.situation_features, fc.opponent_features):
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(LOC_TRAIN_FIRST)], n=150)
    assert bad.empty, bad
SIT_L = fc.situation_features(m, fp)
XLOC = pd.concat([fp[CLASSES + ["same_side"]], SIT_L, fc.opponent_features(m, fp)[["home"]]], axis=1)
XLOC["first_open"] = (((SIT_L["base2"] + SIT_L["base3"]) > 0) & SIT_L["base1"].eq(0)).astype(int)  # 득점권인데 1루 비었음
XLOC = XLOC[LOC_FEAT]
NZ = len(ZONES)
INIT_L = np.log(np.clip(PZ_TRUE, 1e-6, None))
INIT_L -= INIT_L.mean(1, keepdims=True)  # 출발점
yr_l = fp["GYEAR"].to_numpy()
BASE_L = dict(objective="multiclass", num_class=NZ, learning_rate=0.05, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
              verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)



DATE_L = fp["game_date"].to_numpy()
MON_L = np.clip(fp["game_date"].dt.month.to_numpy(), 4, 9)  # 재학습 묶음 달
REG = dict(objective="regression", learning_rate=0.05, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1, verbose=-1, seed=0,
           deterministic=True, force_row_wise=True, num_threads=4, **LOC_SHIFT_LGB)
EX = (fp["x"].to_numpy() - GT[:, 0]) / np.sqrt(GT[:, 2])  # 표준화 잔차 (실제 구종 정규분포 기준)
EZ = (fp["CROSS_PLATE_Z_CM"].to_numpy() - GT[:, 1]) / np.sqrt(GT[:, 3])
TGT = {"ex": EX, "ez": EZ, "u": (EX ** 2 + EZ ** 2) / 2}

def shift_g(g, X, mods):  # 정규분포 g (5개 배열)를 이동 모델로 옮김 -> 새 g
    mx, mz, vx, vz, c = g
    s = np.clip(mods["u"].predict(X), 0.5, 2.0)  # 퍼짐 배수
    return mx + mods["ex"].predict(X) * np.sqrt(vx), mz + mods["ez"].predict(X) * np.sqrt(vz), vx * s, vz * s, c * s

def zone_p(g, q, n=12000):  # 24칸 적분 (나눠서)
    return np.vstack([fc.zone_from_gauss_cov(tuple(np.asarray(a)[i:i + n] for a in g), q.iloc[i:i + n], ZONE_M) for i in range(0, len(q), n)])

def center(L):
    return L - L.mean(1, keepdims=True)

SHIFT_M, BST_L, plan = {}, {}, []
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    iy = np.flatnonzero(yr_l == yr)
    for mon in np.unique(MON_L[iy]):
        start = DATE_L[iy[MON_L[iy] == mon]].min()
        tr = np.flatnonzero((yr_l >= LOC_TRAIN_FIRST) & (DATE_L < start))
        assert DATE_L[tr].max() < start  # 미래 정보 없음
        mods = {k: lgb.train(REG, lgb.Dataset(XLOC.iloc[tr], TGT[k][tr]), LOC_SHIFT[k]) for k in LOC_SHIFT}  # ① 이동 회귀
        SHIFT_M[(yr, mon)] = mods
        P_tr = zone_p(shift_g(tuple(GT[tr].T), XLOC.iloc[tr], mods), fp.iloc[tr])  # ② 학습 공 옮긴 24칸 확률
        BST_L[(yr, mon)] = lgb.train({**BASE_L, **LOC_LGB}, lgb.Dataset(XLOC.iloc[tr], y_zone[tr], init_score=center(np.log(np.clip(P_tr, 1e-6, None)))), LOC_ROUNDS)  # ③
        plan.append({"연도": yr, "월": mon, "학습 초구": len(tr), "예측 시작": str(pd.Timestamp(start).date())})
display(pd.DataFrame(plan).set_index(["연도", "월"]))

def loc_shift_correct(g, q, year, t=None):  # 정규분포 g (구종 t 가정) -> (옮기고 24칸 상황 보정한 24칸 확률 (n, 24), 옮긴 정규분포)
    X = XLOC.loc[q.index].copy()
    if t is not None:
        X.loc[:, CLASSES] = np.tile((np.array(CLASSES) == t).astype(float), (len(X), 1))
    mon = MON_L[fp.index.get_indexer(q.index)]
    P = np.empty((len(q), len(ZONES)))
    G2 = [np.empty(len(q)) for _ in range(5)]
    for mo in np.unique(mon):
        k = mon == mo
        gk = shift_g(tuple(np.asarray(a)[k] for a in g), X[k], SHIFT_M[(year, mo)])
        Pk = zone_p(gk, q[k])
        P[k] = np.exp(log_softmax(BST_L[(year, mo)].predict(X[k], raw_score=True) + center(np.log(np.clip(Pk, 1e-6, None))), axis=1))
        for j in range(5):
            G2[j][k] = gk[j]
    return P, tuple(G2)

미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개


학습 초구       예측 시작
연도   월                   
2019 4  12395  2019-03-23
     5  14963  2019-05-01
     6  17366  2019-06-01
     7  19859  2019-07-02
     8  21650  2019-08-01
     9  24012  2019-09-01
2020 5  25853  2020-05-05
     6  28117  2020-06-02
     7  30496  2020-07-01
     8  32729  2020-08-01
     9  34714  2020-09-01

### 3-3. 진단 (실제 구종 기준)

- model_18 대비 이득: 24칸·좌표 NLL·H/S/C, 달별·주자·아웃

In [12]:
def runner3(q):
    return np.select([(q["base2"] + q["base3"]) > 0, q["base1"] > 0], ["득점권", "1루만"], "없음")

rows = []
for yr, q, msk in [(fc.VAL_YEAR, q_val, val), (fc.TEST_YEAR, q_test, test)]:
    i, yz = np.arange(len(q)), y_zone[msk]
    P24, g24 = loc_shift_correct(tuple(GT[msk].T), q, yr)
    a22 = fc.align(m18, fp, yr)
    ll24 = -np.log(np.clip(P24[i, yz], 1e-12, None))
    co24 = -fc.log_dens(g24, q)
    d = pd.DataFrame({"월": MON_L[msk], "주자": runner3(SIT_L.loc[q.index]), "아웃": SIT_L.loc[q.index, "out_cn"].clip(upper=2).to_numpy(),
                      "24칸 이득": (a22["ll_zone_true"].to_numpy() - ll24) * 1000, "좌표 이득": (a22["nll_coord_true"].to_numpy() - co24) * 1000})
    for var in ["월", "주자", "아웃"]:
        for v, g in d.groupby(var):
            rows.append({"연도": yr, "변수": var, "값": str(v), "공": len(g), "24칸 이득 ×1000": g["24칸 이득"].mean(), "좌표 NLL 이득 ×1000": g["좌표 이득"].mean()})
    rows.append({"연도": yr, "변수": "전체", "값": "", "공": len(d), "24칸 이득 ×1000": d["24칸 이득"].mean(), "좌표 NLL 이득 ×1000": d["좌표 이득"].mean()})
display(pd.DataFrame(rows).set_index(["변수", "값", "연도"]).unstack("연도").round(3))

공          24칸 이득 ×1000        좌표 NLL 이득 ×1000        
연도         2019     2020         2019   2020            2019    2020
변수 값                                                                
아웃 0     4645.0   4375.0        3.496  0.536          16.197   8.489
   1     4500.0   4222.0        1.599  1.566           5.520   6.640
   2     4313.0   4018.0        0.155  0.473           2.300   2.455
월  4     2568.0      NaN        5.090    NaN          12.870     NaN
   5     2403.0   2264.0        2.650  0.380           8.297   3.368
   6     2493.0   2379.0        0.597  1.321           7.192   4.622
   7     1791.0   2233.0        0.977  2.217           4.727   5.731
   8     2362.0   1985.0        1.067  1.834          10.077  10.526
   9     1841.0   3754.0       -0.595 -0.463           3.696   6.054
전체      13458.0  12615.0        1.791  0.861           8.173   5.948
주자 1루만   2886.0   2723.0       -1.764  1.072          -0.123   0.851
   득점권   3346.0   3136.0        3.811  2.990           8.939  10.509
   없음    7226.0   6756.0        2.276 -0.213          11.132   5.886

#### 🔍 3-3 결과: 좌표는 두 해 모두 뚜렷하게, 24칸은 조금 좋아짐
- model_18 대비 이득 ×1000 (실제 구종): 24칸 2019 +1.79 / 2020 +0.86, 좌표 NLL 2019 +8.17 / 2020 +5.95
- 좌표 NLL은 모든 달 + (2020 +3.4 ~ +10.5), 득점권 +8.9 / +10.5, 0아웃 +16.2 / +8.5 -> 0장에서 본 상황 신호(득점권 존 밖, 주자 없을 때 높게)가 좌표에도 반영
- 24칸은 2020 달별 −0.5 ~ +2.2로 섞임, 1루만은 2019 −1.8


## 4. 실전 예측 (예측 구종으로 위치 이어서)

- model_12과 같음, 위치만 3-1에서 고른 값 (분산 따로 포함)

In [13]:
ZONE_BY_TYPE = {}  # 연도 -> (공 키, 구종별 24칸 확률 (n, 9, 24), 구종별 예측 중심 좌우·높이 (n, 9))
def predict(q, year, Pt):  # 초구 q의 공별 예측·점수
    n, i = len(q), np.arange(len(q))
    yt, yz = q["pitch_type"].map(fc.TYPE_IDX).to_numpy(), q["zone"].map(fc.ZONE_IDX).to_numpy()  # 실제 구종·칸 번호
    nll = lambda p: -np.log(np.clip(p, 1e-12, None))  # 공별 log loss (prob_metrics와 같은 방식으로 자름)

    # ① 구종 확률 (n, 9): 2장 결과를 받음

    # ② 위치: 구종 9개 각각 가정 -> 24칸 확률, 실제 위치의 로그 밀도, 예측 중심 (좌우, 높이)
    PZ, LD, MX, MZ = [], [], [], []
    for t in CLASSES:
        P, g = loc_given_type(q, year, t, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL, LOC_SEASON_F, LOC_AC, LOC_VAR)
        P, g = loc_shift_correct(g, q, year, t)  # 중심·퍼짐 상황 이동 + 24칸 상황 보정 (model_20), 좌표도 옮긴 정규분포
        PZ.append(P); LD.append(fc.log_dens(g, q)); MX.append(g[0]); MZ.append(g[1])
    PZ, LD, MX, MZ = np.stack(PZ, 1), np.stack(LD, 1), np.stack(MX, 1), np.stack(MZ, 1)  # (n, 구종, ...)
    ZONE_BY_TYPE[year] = (q[fc.KEYS].reset_index(drop=True), PZ, MX, MZ)  # 구종별 24칸 확률·예측 중심 (서비스 화면용, 결과 파일에 따로 저장)
    joint = Pt[:, :, None] * PZ  # 구종 확률 × 그 구종일 때 칸 확률 = 구종×칸 결합 확률 (n, 9, 24)
    P_zone = joint.sum(1)  # 실전 24칸 확률 = 구종으로 합침
    # 좌표: 구종 확률로 섞은 정규분포에서 실제 위치의 로그 밀도
    ld_mix = np.log(np.einsum("nt,nt->n", Pt, np.exp(LD - LD.max(1, keepdims=True)))) + LD.max(1)  #  공마다 최댓값 빼기 -> exp -> 섞기 -> log (계산 안전)
    dist_true = np.hypot(q["x"] - MX[i, yt], q["CROSS_PLATE_Z_CM"] - MZ[i, yt])  # 실제 구종 중심과의 거리 cm
    dist_mix = np.hypot(q["x"] - (Pt * MX).sum(1), q["CROSS_PLATE_Z_CM"] - (Pt * MZ).sum(1))  # 구종 확률로 가중한 중심과의 거리 cm

    t_hat = np.array(CLASSES)[Pt.argmax(1)]  # 1등 구종 이름

    # 공별 결과 표 (필수: 키, 구종 확률 9개, 실전 24칸 확률 / 나머지는 선택 열)
    return pd.concat([
        q[["gameID", "pit_no", "pitcher_code", "GYEAR", "pitch_type", "zone"]].reset_index(drop=True),
        pd.DataFrame(Pt, columns=fc.P_TYPE),  # p_type_<구종>
        pd.DataFrame(P_zone, columns=fc.P_ZONE),  # p_zone_<칸>
        pd.DataFrame({"t_hat": t_hat}),  # 1등 구종 이름
        pd.DataFrame({
            "ll_type": nll(Pt[i, yt]),  # 구종 log loss
            "ll_zone": nll(P_zone[i, yz]),  # 24칸 log loss (실전)
            "ll_zone_true": nll(PZ[i, yt, yz]),  # 24칸 log loss (실제 구종)
            "ll_joint": nll(joint[i, yt, yz]),  # 구종×칸 log loss
            "nll_coord": -ld_mix,  # 좌표 NLL (실전)
            "nll_coord_true": -LD[i, yt],  # 좌표 NLL (실제 구종)
            "dist": np.asarray(dist_mix),  # 좌표 거리 cm (실전)
            "dist_true": np.asarray(dist_true),  # 좌표 거리 cm (실제 구종)
        })], axis=1)

In [14]:
# 2019 검증 + 2020 평가 초구 전부 예측
pp = pd.concat([predict(q_val, fc.VAL_YEAR, P_TYPE_ALL[val]), predict(q_test, fc.TEST_YEAR, P_TYPE_ALL[test])], ignore_index=True)
print(pp.shape)
# 확인: 구종은 model_18와 같아야 함 (위치는 24칸·좌표 모두 바뀜)
chk = pp.merge(m18, on=fc.KEYS, suffixes=("", "_18"))
same = ["ll_type"] + fc.P_TYPE
bad = [c for c in same if not np.allclose(chk[c], chk[c + "_18"], equal_nan=True, atol=1e-6)]
assert len(chk) == len(pp) and not bad, f"model_18와 다른 열: {bad}"
print("구종 = model_18 (재현 확인), 위치(24칸·좌표)만 바뀜")

(26073, 48)
구종 = model_18 (재현 확인), 위치(24칸·좌표)만 바뀜


## 5. 평가

- 채점은 `fc.evaluate`, model_18와 나란히 / 경기 단위 부트스트랩 (model_20 − model_18, 2020), H/S/C(`ll_hsc`)도 나빠지지 않아야 함 (채택 기준 (다))
- 2020 너클 뺀 값도 봄

In [15]:
def eval_years(pp_):  # 2019·2020 주요 지표 표
    return pd.DataFrame({"검증 2019": fc.evaluate(pp_, fp, fc.VAL_YEAR), "평가 2020": fc.evaluate(pp_, fp, fc.TEST_YEAR)})

summary = eval_years(pp)
m18_sum = eval_years(m18)
display(pd.concat({MODEL: summary, "model_18": m18_sum, "차이 (model_18 대비)": summary - m18_sum}, axis=1).round(4))

model_20          model_18          차이 (model_18 대비)        
                     검증 2019  평가 2020  검증 2019  평가 2020          검증 2019 평가 2020
구종 logloss            1.0613   1.0561   1.0613   1.0561           0.0000  0.0000
구종 top1               0.5385   0.5332   0.5385   0.5332           0.0000  0.0000
구종 top2               0.8177   0.8218   0.8177   0.8218           0.0000  0.0000
구종 top3               0.9374   0.9412   0.9374   0.9412           0.0000  0.0000
구종 macro F1           0.3226   0.3545   0.3226   0.3545           0.0000  0.0000
24칸 logloss (실전)      2.9258   2.9258   2.9278   2.9266          -0.0020 -0.0008
24칸 top1              0.1397   0.1372   0.1384   0.1371           0.0013  0.0002
24칸 top3              0.3068   0.3074   0.3041   0.3066           0.0027  0.0008
H/S/C logloss         1.0719   1.0728   1.0723   1.0728          -0.0003 -0.0001
H/S/C 정확도             0.4013   0.4015   0.4019   0.4024          -0.0006 -0.0009
24칸 logloss (실제 구종)   2.8660   2.8694   2.8678   2.8703          -0.0018 -0.0009
구종×칸 logloss          3.9273   3.9255   3.9291   3.9264          -0.0018 -0.0009
좌표 평균거리 cm           32.3027  31.5755  32.3944  31.6319          -0.0917 -0.0564
좌표 NLL                9.2870   9.2430   9.2949   9.2493          -0.0079 -0.0063

In [16]:
display(fc.bootstrap_diff(fc.pitch_scores(m18, fp), fc.pitch_scores(pp, fp)).round(4))  # 음수 = 이 모델이 좋음 (b − a)
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    a20, a22 = fc.align(m18, fp, yr), fc.align(pp, fp, yr)  # a20 = 비교 대상 model_18
    nk = a20["pitch_type"].ne("KNUC").to_numpy()
    d_true = a22["ll_zone_true"].to_numpy() - a20["ll_zone_true"].to_numpy()
    print(f"{yr} 24칸 실제 구종 차이 (model_20 − model_18): 전체 {d_true.mean():+.5f}, 너클 뺌 {d_true[nk].mean():+.5f}")

,차이 (b−a),95% 하한,95% 상한,판정
ll_type,0.0000,0.0000,0.0000,차이 불확실
ll_zone,-0.0008,-0.0017,0.0001,차이 불확실
ll_hsc,-0.0001,-0.0004,0.0004,차이 불확실
ll_joint,-0.0009,-0.0019,0.0002,차이 불확실
nll_coord,-0.0063,-0.0082,-0.0044,좋아짐
dist,-0.0564,-0.0799,-0.0314,좋아짐


2019 24칸 실제 구종 차이 (model_20 − model_18): 전체 -0.00179, 너클 뺌 -0.00179
2020 24칸 실제 구종 차이 (model_20 − model_18): 전체 -0.00086, 너클 뺌 -0.00087


### 🥇1등🥇
#### 2020 좌표 NLL −0.0063 · 거리 −0.06cm (부트스트랩 좋아짐), 24칸 실전 −0.0008 · H/S/C 나빠짐 없음 -> 채택
- 구종은 model_18와 같음 (재현 확인), 위치 정규분포 중심·퍼짐을 상황별로 옮기고 그 위에 24칸 상황 보정
- 2019: 좌표 NLL −0.0079, 24칸 실전 −0.0020 / 2020: 좌표 NLL −0.0063, 24칸 실전 −0.0008 (95% −0.0017 ~ +0.0001)
- 이동 회귀 트리 수는 2018 안에서 정함 (2019로 안 고름) -> 2019 숫자가 안 부풀려짐, 좌표 이득이 2020에도 대부분 이어짐
- H/S/C log loss 차이 불확실 (채택 기준 (다) 통과), H/S/C 정확도 −0.0006 / −0.0009
- 위치 LightGBM이 4개 (이동 회귀 3개 + 24칸 보정), 모두 월 단위 재학습
- **채택 -> 다음 비교 출발점** (사용자 결정 2026-11-12)

## 6. 저장

- `results/model_20/` (채택하면 `compare.ipynb` `ADOPTED`에 추가)

In [17]:
fc.save_results(MODEL, pp, {"summary": summary})

저장: results\model_20 ['per_pitch.parquet', 'summary.csv']


- 서비스 화면용 (2026-11-12 추가): 공마다 구종 9개 각각을 가정한 24칸 확률과 예측 중심(좌우·높이 cm)을 `results/model_20/zone_by_type.parquet`에 따로 저장


In [18]:
parts = []
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    keys, PZ_, MX_, MZ_ = ZONE_BY_TYPE[yr]
    cols = {f"pz_{t}_{z}": PZ_[:, j, k] for j, t in enumerate(CLASSES) for k, z in enumerate(ZONES)}
    cols.update({f"cx_{t}": MX_[:, j] for j, t in enumerate(CLASSES)})
    cols.update({f"cz_{t}": MZ_[:, j] for j, t in enumerate(CLASSES)})
    parts.append(pd.concat([keys, pd.DataFrame(cols)], axis=1))
zbt = pd.concat(parts, ignore_index=True)
chk_ = zbt.merge(pp[fc.KEYS + fc.P_TYPE + fc.P_ZONE], on=fc.KEYS)
mix = np.einsum("nt,ntz->nz", chk_[fc.P_TYPE].to_numpy(), chk_[[f"pz_{t}_{z}" for t in CLASSES for z in ZONES]].to_numpy().reshape(len(chk_), 9, 24))
assert np.allclose(mix, chk_[fc.P_ZONE].to_numpy(), atol=1e-8), "구종별 확률을 섞은 값이 실전 24칸과 다름"
zbt.to_parquet("results/model_20/zone_by_type.parquet", index=False)
print("저장: results/model_20/zone_by_type.parquet", zbt.shape, "(구종별로 섞으면 실전 24칸과 같음, 확인)")


저장: results/model_20/zone_by_type.parquet (26073, 236) (구종별로 섞으면 실전 24칸과 같음, 확인)
